## LEVEL 2
1. Predict google place type from business name (if primaryType is non-suggestive)
2. Assign cuisineType for each place where possible - rest unspecified
3. Assign venueType for each place
4. Export places by resolved and unresolved

#### Initialize

In [1]:
import sys
from pathlib import Path
import pandas as pd

HERE = Path.cwd()
PARENT = HERE.parent.parent.parent  # server/scripts
if str(PARENT) not in sys.path:
    sys.path.insert(0, str(PARENT))
LEVEL1_BUCKET_PATH = PARENT / "server/out/places_level1"
LEVEL2_BUCKET_PATH = PARENT / "server/out/places_level2"
LEVEL2_BUCKET_PATH.mkdir(parents=True, exist_ok=True)
LEVEL1_BUCKET = [f for f in LEVEL1_BUCKET_PATH.rglob("*.csv") if f.is_file()]
DF_LEVEL1 = pd.concat([pd.read_csv(f) for f in LEVEL1_BUCKET], ignore_index=True)

#### Parse Types

In [2]:
from server.scripts.clean_places_level_2.CHAIN_REGISTER import predict_google_type_from_chain, find_block_chain_name
from server.scripts.clean_places_level_2.type_parsing import parse_type, check_takeaway, predict_cuisine_from_name

df_level2 = DF_LEVEL1.copy()
df_level2["is_chain"] = df_level2["displayName"].apply(lambda n: bool(find_block_chain_name(str(n or ""))))
df_level2["predictedType"] = df_level2.apply(predict_google_type_from_chain, axis=1)
mask = df_level2["predictedType"].eq("")
df_level2.loc[mask, "predictedType"] = df_level2.loc[mask].apply(predict_cuisine_from_name, axis=1)
df_level2["cuisineType"] = df_level2.apply(parse_type, axis=1)
df_level2["venueType"] = df_level2.apply(check_takeaway, axis=1)

# ── Diagnostics ───────────────────────────────────────────────────────────────
dist = df_level2["cuisineType"].value_counts()
unresolved = (df_level2["cuisineType"] == "Unspecified").sum()
chains = df_level2["is_chain"].sum()
print(f"Chains detected      : {chains} / {len(df_level2)}  ({chains/len(df_level2):.1%})")
print(f"Unique cuisineTypes  : {dist.nunique()}")
print(f"Still 'Unspecified'  : {unresolved} / {len(df_level2)}  ({unresolved/len(df_level2):.1%})")

Chains detected      : 1268 / 13092  (9.7%)
Unique cuisineTypes  : 40
Still 'Unspecified'  : 1607 / 13092  (12.3%)


In [3]:
unspecified = df_level2[df_level2["cuisineType"]=="Unspecified"]
sample = unspecified[["displayName", "primaryType", "primaryTypeDisplayName", "types", "cuisineType"]]
sample.sample(10)["displayName"].to_list()

['Chicken Express',
 'Ayannas London',
 'The Grand Undaal',
 'Sabor Tecleño',
 'Baha Lounge',
 'Uzbek Corner',
 'Palace tandoori w6',
 'Smoke & Pepper (Earls Court)',
 'Drunch Notting Hill',
 'Kokodoo - Fulham Broadway']

#### Export

In [3]:
df_resolved = df_level2[df_level2["cuisineType"]!="Unspecified"]
df_resolved.to_csv(LEVEL2_BUCKET_PATH / "places_resolved.csv", index=False)
df_unresolved = df_level2[df_level2["cuisineType"]=="Unspecified"]
df_unresolved.to_csv(LEVEL2_BUCKET_PATH / "places_unresolved.csv", index=False)

In [4]:
# df_resolved.reset_index(drop=True, inplace=True)
# df_resolved[df_resolved['cuisineType']=='Bar & Pub'][['displayName', 'primaryTypeDisplayName', 'primaryType', 'types']]
df_unresolved

,id,displayName,primaryTypeDisplayName,rating,userRatingCount,location,shortFormattedAddress,googleMapsUri,priceRange,priceLevel,...,addressDescriptor,postalAddress,tile_id,tile_path_id,seed_index,level,is_chain,predictedType,cuisineType,venueType
3,ChIJ-YLGMkUDdkgRdFYzsUYD3dU,Thurner's,Restaurant,5.0,67.0,"{'latitude': 51.512034199999995, 'longitude': ...","27 Garlick Hill, London",https://maps.google.com/?cid=15410477102087231...,"{'startPrice': {'currencyCode': 'GBP', 'units'...",NaN,...,{'landmarks': [{'name': 'places/ChIJ_wnAfqoEdk...,"{'regionCode': 'GB', 'languageCode': 'en-US', ...",8b194ad304d3fff,0-3-2-3,0,3,False,,Unspecified,Dine-In
23,ChIJ61pQaH8DdkgRimqN0XUuGnM,The Wolseley City,Restaurant,4.4,337.0,"{'latitude': 51.5110508, 'longitude': -0.0866166}","HOUSE OF FRASER, Underground Ltd, 68 King Will...",https://maps.google.com/?cid=82939927473179470...,NaN,NaN,...,{'landmarks': [{'name': 'places/ChIJw6LfxVMDdk...,"{'regionCode': 'GB', 'languageCode': 'en-US', ...",8a194ad3044ffff,0-1-1,0,2,False,,Unspecified,Dine-In
29,ChIJ83IDLVUbdkgRMN0SQY9E3Uo,Rustichino,Restaurant,4.1,52.0,"{'latitude': 51.515091299999995, 'longitude': ...",London,https://maps.google.com/?cid=53945433107224896...,"{'startPrice': {'currencyCode': 'GBP', 'units'...",NaN,...,{'landmarks': [{'name': 'places/ChIJu9NKyqoEdk...,NaN,8a194ad30497fff,0-2-2,0,2,False,,Unspecified,Dine-In
36,ChIJAQAQJFMDdkgRWSgkDpXi2Zc,Socialize at Threadneedles,Restaurant,4.0,1.0,"{'latitude': 51.514032099999994, 'longitude': ...","5 Threadneedle St, London",https://maps.google.com/?cid=10942025899488585...,NaN,NaN,...,{'landmarks': [{'name': 'places/ChIJCQxrJlMDdk...,"{'regionCode': 'GB', 'languageCode': 'en-US', ...",8a194ad3054ffff,0-5-1,0,2,False,,Unspecified,Dine-In
37,ChIJAQDAwFQDdkgR204T_jVXXdg,The Parlour,Restaurant,4.4,70.0,"{'latitude': 51.513632, 'longitude': -0.0901102}","27 Poultry, London",https://maps.google.com/?cid=15590713374434086...,"{'startPrice': {'currencyCode': 'GBP', 'units'...",NaN,...,{'landmarks': [{'name': 'places/ChIJTUEYl1QDdk...,"{'regionCode': 'GB', 'languageCode': 'en-US', ...",8a194ad3040ffff,0-0-1,0,2,False,,Unspecified,Dine-In
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13043,ChIJUzOTY1YDdkgR0sly4nyDPq0,Scaldy,Restaurant,5.0,10.0,"{'latitude': 51.4738178, 'longitude': -0.0679974}","Red Bull, 116 Peckham High St, London",https://maps.google.com/?cid=12483559789515950...,"{'startPrice': {'currencyCode': 'GBP', 'units'...",NaN,...,{'landmarks': [{'name': 'places/ChIJ868HkwoDdk...,"{'regionCode': 'GB', 'languageCode': 'en-US', ...",89194ad0607ffff,98-1,98,1,False,,Unspecified,Dine-In
13048,ChIJ_b9GXXUDdkgRyzg4KbGIg_8,KHF Peckham,Takeout Restaurant,4.4,192.0,"{'latitude': 51.4735995, 'longitude': -0.0716955}","35 Peckham High St, London",https://maps.google.com/?cid=18411709996102858...,"{'startPrice': {'currencyCode': 'GBP', 'units'...",PRICE_LEVEL_INEXPENSIVE,...,{'landmarks': [{'name': 'places/ChIJJfx5ZXUDdk...,"{'regionCode': 'GB', 'languageCode': 'en-US', ...",8a194ad060d7fff,98-3-2,98,2,False,,Unspecified,Takeaway
13056,ChIJo1umXnUDdkgRsB-sZFil4pA,Tiwa 'N' Tiwa,Restaurant,4.3,234.0,"{'latitude': 51.4733076, 'longitude': -0.07145...","34 Peckham High St, London",https://maps.google.com/?cid=10440088685262938...,"{'startPrice': {'currencyCode': 'GBP', 'units'...",NaN,...,{'landmarks': [{'name': 'places/ChIJJfx5ZXUDdk...,"{'regionCode': 'GB', 'languageCode': 'en-US', ...",8a194ad060c7fff,98-3-0,98,2,False,,Unspecified,Dine-In
13058,ChIJs4zVSAADdkgR-9HZ2YDUaiA,Annapurna Spice,Restaurant,4.2,37.0,"{'latitude': 51.4723587, 'longitude': -0.0694624}","Front of Boots, Market Place, Unit 10, The Ayl...",https://maps.google.com/?cid=23359130066208568...,"{'startPrice': {'currencyCode': 'GBP', 'units'...",NaN,...,{'landmarks': [{'name': 'places/ChIJUcj3dKADdk...,"{'regionCode': 'GB', 'languageCode': 'en-US', ...",8a194ad060cffff,98-3-1,98,2,False,,Unspecified,Dine-In
